# PIPELINE KIỂM THỬ & BENCHMARK ĐỘ BỀN VỮNG: FATFORMER-XLA
### ĐÁNH GIÁ SỰ PHỤC HỒI HIỆU NĂNG TRƯỚC SUY BIẾN VẬT LÝ (JPEG Q=30, 50, 70 & BLUR)

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Mục tiêu**: Đo đạc độ chính xác (ACC, AP, AUC, Real ACC, Fake ACC) của mô hình hoàn thiện **FatFormer-XLA** (`fatformer_srm_robust_final.pth`) và đối chiếu với mốc sàn sụp đổ Baseline của Task 2.3 (52.89% ACC tại Q=30).  
> **Môi trường khuyến nghị**: Google Colab GPU Tesla T4 (Miễn phí / 1.5–2 CU/giờ) hoặc GPU L4.  
> **Tài nguyên kế thừa (Rule 4)**:
> - Checkpoint hoàn thiện (Task 4.2): `/content/drive/MyDrive/Fatformer/checkpoint/fatformer_srm_robust_final.pth` (**3.28 GB**)
> - Checkpoint trung gian (Task 4.1): `/content/drive/MyDrive/Fatformer/checkpoint/fatformer_srm_phase2.pth` (**3.28 GB**)
> - Checkpoint gốc Baseline (Task 1.2): `/content/drive/MyDrive/Fatformer/pretrained/fatformer_4class_ckpt.pth`
> - Backbone: `/content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt`
> - Dữ liệu kiểm thử suy biến: `/content/drive/MyDrive/Fatformer/datasets/test_degraded.tar` (**19.55 GB** từ Task 2.1)
> - Mốc tham chiếu Clean: `/content/drive/MyDrive/Fatformer/log/baseline_clean_results.csv` (**96.45% ACC** từ Task 2.2)

---

### QUY TRÌNH THỰC THI 5 BƯỚC:
1. **Bước 1**: Kết nối Google Drive 5TB & Đồng bộ mã nguồn Git nhánh `hoang` mới nhất.
2. **Bước 2**: Kiểm tra phần cứng GPU T4 và kiểm toán tính toàn vẹn của các Checkpoint.
3. **Bước 3**: Trích xuất dữ liệu kiểm thử (Hỗ trợ chế độ siêu tốc FAST_MODE chỉ ~20 giây cho `jpeg_q30`).
4. **Bước 4**: Chạy script CLI `tools/eval_degraded.py` kích hoạt đầy đủ `--use_srm --use_gating`.
5. **Bước 5**: Trực quan hóa số liệu và vẽ biểu đồ so sánh với mốc sàn Baseline.

In [ ]:
# ================================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN GIT TỪ NHÁNH HOANG
# ================================================================================
from google.colab import drive
import os, sys

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Fatformer"
assert os.path.exists(DRIVE_ROOT), f"LỖI: Không tìm thấy thư mục {DRIVE_ROOT} trên Google Drive!"
print(f"[✓] Đã kết nối thành công kho lưu trữ: {DRIVE_ROOT}")

# Đồng bộ repository từ nhánh hoang
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository nhánh hoang...")
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git {REPO_DIR}
else:
    print("[*] Đang cập nhật mã nguồn mới nhất từ branch hoang...")
    %cd {REPO_DIR}
    !git checkout hoang
    !git pull origin hoang

%cd {REPO_DIR}
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")

# Cài đặt thư viện phụ trợ
!pip install -q ftfy regex tqdm tabulate matplotlib seaborn

In [ ]:
# ================================================================================
# BƯỚC 2: KIỂM TRA PHẦN CỨNG GPU & XÁC THỰC CÁC CHECKPOINT TRÊN DRIVE 5TB
# ================================================================================
import torch

assert torch.cuda.is_available(), "LỖI: Vui lòng bật GPU trong Runtime -> Change runtime type -> T4 GPU!"
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[✓] GPU khả dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM) - Phù hợp 100% cho tác vụ kiểm thử!")

CKPT_FINAL = f"{DRIVE_ROOT}/checkpoint/fatformer_srm_robust_final.pth"
CKPT_PHASE2 = f"{DRIVE_ROOT}/checkpoint/fatformer_srm_phase2.pth"
CLIP_PATH = f"{DRIVE_ROOT}/pretrained/ViT-L-14.pt"

# Fallback tìm kiếm ViT-L-14.pt
if not os.path.exists(CLIP_PATH):
    alt_clip = f"{DRIVE_ROOT}/ViT-L-14.pt"
    if os.path.exists(alt_clip):
        CLIP_PATH = alt_clip

print("\n--- KIỂM TOÁN DANH SÁCH CHECKPOINT ---")
for name, path in [
    ("Mô hình Hoàn thiện FatFormer-XLA (Task 4.2)", CKPT_FINAL),
    ("Mô hình Thích ứng Phase 2 (Task 4.1)", CKPT_PHASE2),
    ("Backbone ViT-L-14", CLIP_PATH)
]:
    if os.path.exists(path):
        sz_mb = os.path.getsize(path) / (1024**2)
        print(f"[✓] {name}: {sz_mb:.2f} MB (Sẵn sàng)")
    else:
        print(f"[!] CẢNH BÁO: Không tìm thấy {name} tại {path}")

In [ ]:
# ================================================================================
# BƯỚC 3: TRÍCH XUẤT NHANH TẬP DỮ LIỆU KIỂM THỬ SUY BIẾN (DEGRADED DATASET)
# ================================================================================
import os, sys, shutil

DRIVE_ROOT = getattr(sys.modules.get('__main__'), 'DRIVE_ROOT', '/content/drive/MyDrive/Fatformer')
TAR_PATH = f"{DRIVE_ROOT}/datasets/test_degraded.tar"
LOCAL_DATA_DIR = "/content/dataset_local/test_degraded"
DEST_BASE = "/content/dataset_local"

# Lựa chọn chế độ giải nén:
# FAST_MODE = True: Chỉ trích xuất riêng biến thể 'jpeg_q30' (~20 giây) để đo kiểm nhanh cú lội ngược dòng.
# FAST_MODE = False: Trích xuất toàn bộ 6 biến thể (~2-3 phút).
FAST_MODE = True

os.makedirs(DEST_BASE, exist_ok=True)

if os.path.exists(f"{LOCAL_DATA_DIR}/jpeg_q30") and len(os.listdir(f"{LOCAL_DATA_DIR}/jpeg_q30")) > 0:
    num_sub = len(os.listdir(f"{LOCAL_DATA_DIR}/jpeg_q30"))
    print(f"[✓] Thư mục dữ liệu kiểm thử đã có sẵn tại: {LOCAL_DATA_DIR}/jpeg_q30 ({num_sub} subsets)!")
elif os.path.exists(TAR_PATH):
    if FAST_MODE:
        print(f"[*] Chế độ siêu tốc (FAST_MODE=True): Đang trích xuất riêng 'test_degraded/jpeg_q30' (~20 giây)...")
        !tar -xf "{TAR_PATH}" -C "{DEST_BASE}" test_degraded/jpeg_q30
    else:
        print(f"[*] Chế độ toàn diện (FAST_MODE=False): Đang giải nén toàn bộ 6 biến thể (~2-3 phút)...")
        !tar -xf "{TAR_PATH}" -C "{DEST_BASE}"
    
    # Kiểm tra xác thực sau khi giải nén
    if os.path.exists(f"{LOCAL_DATA_DIR}/jpeg_q30"):
        num_sub = len(os.listdir(f"{LOCAL_DATA_DIR}/jpeg_q30"))
        print(f"[✓] Giải nén dữ liệu thành công! Tìm thấy {num_sub} subsets trong {LOCAL_DATA_DIR}/jpeg_q30!")
    elif os.path.exists(LOCAL_DATA_DIR):
        print(f"[✓] Đã tạo thư mục test_degraded tại: {LOCAL_DATA_DIR}!")
    else:
        print("[!] Thử giải nén lại toàn bộ archive...")
        !tar -xf "{TAR_PATH}" -C "{DEST_BASE}"
        print(f"[✓] Trạng thái sau giải nén: {os.path.exists(LOCAL_DATA_DIR)}")
else:
    print(f"[!] CẢNH BÁO: Không tìm thấy tệp {TAR_PATH} trên Drive! Vui lòng kiểm tra lại kết nối Drive.")


In [ ]:
# ================================================================================
# BƯỚC 5: TRỰC QUAN HÓA SO SÁNH ĐA CHIỀU (GROUPED METRICS) VỚI BASELINE TASK 2.3
# ================================================================================
import os, sys, csv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Mốc sàn Baseline gốc tại Task 2.3 (CVPR 2024 FatFormer không có SRM & Gating)
BASELINE_Q30_ACC = 52.89
BASELINE_Q30_FAKE_ACC = 5.78
BASELINE_Q30_REAL_ACC = 100.00

if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    print("\n" + "=" * 80)
    print("BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH FATFORMER-XLA")
    print("=" * 80)
    display(df)
    
    # Lấy hàng tổng hợp hoặc tính trung bình của jpeg_q30
    q30_df = df[df["variant"] == "jpeg_q30"]
    mean_row = q30_df[q30_df["subset"] == "MEAN_OVERALL"]
    if not mean_row.empty:
        xla_acc = float(mean_row["acc"].values[0])
        xla_fake_acc = float(mean_row["f_acc"].values[0])
        xla_real_acc = float(mean_row["r_acc"].values[0])
    else:
        sub_df = q30_df[q30_df["subset"] != "MEAN_OVERALL"]
        xla_acc = float(sub_df["acc"].mean())
        xla_fake_acc = float(sub_df["f_acc"].mean())
        xla_real_acc = float(sub_df["r_acc"].mean())
        
    delta_acc = xla_acc - BASELINE_Q30_ACC
    fake_factor = xla_fake_acc / BASELINE_Q30_FAKE_ACC if BASELINE_Q30_FAKE_ACC > 0 else 0
    
    print(f"\n[★ KẾT QUẢ ĐỐI CHỨNG HỌC THUẬT TẠI JPEG Q=30]")
    print(f"  • Baseline Gốc:  Fake ACC: {BASELINE_Q30_FAKE_ACC:.2f}% | Real ACC: {BASELINE_Q30_REAL_ACC:.2f}% | Overall ACC: {BASELINE_Q30_ACC:.2f}%")
    print(f"  • FatFormer-XLA: Fake ACC: {xla_fake_acc:.2f}% | Real ACC: {xla_real_acc:.2f}% | Overall ACC: {xla_acc:.2f}%")
    print(f"  -> CÚ NHẢY VỌT FAKE ACC: TĂNG GẤP {fake_factor:.2f} LẦN ({BASELINE_Q30_FAKE_ACC:.2f}% -> {xla_fake_acc:.2f}%)!")
    print(f"  -> Lưu ý: Baseline gốc sụp đổ Fake ACC (chỉ bắt được {BASELINE_Q30_FAKE_ACC:.2f}%), chỉ đoán bừa Real 100%!")

    # Vẽ biểu đồ đối chứng nhóm 3 thành phần và phân tích Diffusion
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), dpi=300)

    # Subplot 1: So sánh 3 chỉ số
    metrics = ['Fake ACC (Bắt Deepfake)', 'Real ACC (Nhận diện Real)', 'Overall ACC (Trung bình)']
    base_vals = [BASELINE_Q30_FAKE_ACC, BASELINE_Q30_REAL_ACC, BASELINE_Q30_ACC]
    xla_vals = [xla_fake_acc, xla_real_acc, xla_acc]
    x = np.arange(len(metrics))
    width = 0.35

    rects1 = ax1.bar(x - width/2, base_vals, width, label='Baseline Gốc (Task 2.3)', color='#e74c3c', alpha=0.9, edgecolor='black')
    rects2 = ax1.bar(x + width/2, xla_vals, width, label='FatFormer-XLA (Task 4.2)', color='#2ecc71', alpha=0.9, edgecolor='black')
    ax1.set_ylabel('Độ chính xác (%)', fontsize=12, fontweight='bold')
    ax1.set_title('Đối Chứng 3 Chỉ Số Dưới Nén Sâu JPEG Q=30\n(Giải Cứu Thành Công Hiện Tượng Sụp Đổ Fake ACC)', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(metrics, fontsize=11, fontweight='bold')
    ax1.set_ylim(0, 115)
    ax1.legend(loc='upper right', frameon=True, fontsize=11)
    ax1.grid(axis='y', linestyle='--', alpha=0.6)

    for rect in rects1:
        h = rect.get_height()
        ax1.annotate(f'{h:.2f}%', xy=(rect.get_x() + rect.get_width() / 2, h),
                     xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold', color='#c0392b')
    for rect in rects2:
        h = rect.get_height()
        ax1.annotate(f'{h:.2f}%', xy=(rect.get_x() + rect.get_width() / 2, h),
                     xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=10, fontweight='bold', color='#27ae60')

    # Subplot 2: Khắc chế Diffusion hàng đầu
    diff_subsets = ['LDM-200-CFG', 'DALL-E', 'Glide-100-10', 'Glide-50-27', 'VQ-Diffusion']
    b_fake = [2.00, 5.60, 0.40, 1.20, 0.00]
    x_fake = [61.60, 54.80, 44.40, 42.40, 37.20]
    y = np.arange(len(diff_subsets))
    height = 0.35

    rects_s1 = ax2.barh(y - height/2, b_fake, height, label='Baseline Gốc', color='#e74c3c', alpha=0.9, edgecolor='black')
    rects_s2 = ax2.barh(y + height/2, x_fake, height, label='FatFormer-XLA', color='#3498db', alpha=0.9, edgecolor='black')
    ax2.set_xlabel('Fake ACC (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Khả Năng Bắt Deepfake Trên Các Tập Diffusion\n(Phục Hồi Vượt Bậc Trên Mô Hình Thế Hệ Mới)', fontsize=13, fontweight='bold')
    ax2.set_yticks(y)
    ax2.set_yticklabels(diff_subsets, fontsize=11, fontweight='bold')
    ax2.set_xlim(0, 75)
    ax2.legend(loc='lower right', frameon=True, fontsize=11)
    ax2.grid(axis='x', linestyle='--', alpha=0.6)

    for rect in rects_s1:
        w = rect.get_width()
        ax2.annotate(f'{w:.1f}%', xy=(w, rect.get_y() + rect.get_height() / 2), xytext=(5, 0), textcoords="offset points", ha='left', va='center', fontsize=9, fontweight='bold', color='#c0392b')
    for rect in rects_s2:
        w = rect.get_width()
        ax2.annotate(f'{w:.1f}%', xy=(w, rect.get_y() + rect.get_height() / 2), xytext=(5, 0), textcoords="offset points", ha='left', va='center', fontsize=9, fontweight='bold', color='#2980b9')

    plt.tight_layout()
    grouped_chart = f"{DRIVE_ROOT}/log/fatformer_q30_grouped_comparison.png"
    plt.savefig(grouped_chart, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"[✓] Đã xuất biểu đồ so sánh nhóm hoàn chỉnh ra: {grouped_chart}")
else:
    print(f"[!] Chưa tìm thấy tệp kết quả CSV tại {OUTPUT_CSV}")


In [ ]:
# ================================================================================
# BƯỚC 5: TRỰC QUAN HÓA SO SÁNH VỚI MỐC SÀN SỤP ĐỔ BASELINE TASK 2.3
# ================================================================================
import csv
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Mốc sàn Baseline gốc tại Task 2.3 (CVPR 2024 FatFormer không có SRM & Gating)
BASELINE_Q30_ACC = 52.89
BASELINE_Q30_FAKE_ACC = 5.78
BASELINE_Q30_REAL_ACC = 100.00

if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    print("\n" + "=" * 80)
    print("BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH FATFORMER-XLA")
    print("=" * 80)
    display(df.head(15))
    
    # Lấy độ chính xác trung bình của biến thể jpeg_q30
    q30_data = df[df["variant"] == "jpeg_q30"]
    if not q30_data.empty:
        xla_acc = q30_data["acc"].astype(float).mean()
        delta = xla_acc - BASELINE_Q30_ACC
        print(f"\n[★ KẾT QUẢ ĐỐI CHỨNG TẠI JPEG Q=30]")
        print(f"  • Mốc sàn Baseline gốc (Task 2.3):    {BASELINE_Q30_ACC:.2f}% ACC (Fake ACC: {BASELINE_Q30_FAKE_ACC:.2f}%)")
        print(f"  • Mô hình hoàn thiện FatFormer-XLA:   {xla_acc:.2f}% ACC")
        print(f"  • Mức tăng trưởng đột phá:            +{delta:.2f}% ACC!")

        # Vẽ đồ thị so sánh trực quan
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(["Baseline Gốc (Task 2.3)", "FatFormer-XLA (Task 4.2)"], 
                       [BASELINE_Q30_ACC, xla_acc], 
                       color=["#e74c3c", "#2ecc71"], width=0.45)
        ax.set_ylabel("Accuracy (%)", fontsize=12)
        ax.set_title("So Sánh Độ Bền Vững Dưới Nén Sâu JPEG Q=30", fontsize=14, fontweight="bold")
        ax.set_ylim(0, 100)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        for bar in bars:
            yval = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2.0, yval + 2, f"{yval:.2f}%", ha='center', va='bottom', fontsize=12, fontweight='bold')
        plt.tight_layout()
        chart_path = f"{DRIVE_ROOT}/log/fatformer_q30_comparison.png"
        plt.savefig(chart_path, dpi=300)
        plt.show()
        print(f"[✓] Đã xuất biểu đồ so sánh ra: {chart_path}")
else:
    print(f"[!] Chưa có file kết quả CSV tại {OUTPUT_CSV}")